# **Agentic AI-Based Travel Planning Assistant Using LangChain**

## **Weather Tool (Open-Meteo API)**

In [6]:
import os

os.makedirs("data", exist_ok=True)
os.makedirs("tools", exist_ok=True)

print("Folders ready")

Folders ready


In [7]:
!pip install requests

In [8]:
import requests

def get_weather(latitude: float, longitude: float):
    """
    Fetch daily maximum temperature from Open-Meteo API.
    """

    url = (
        "https://api.open-meteo.com/v1/forecast"
        f"?latitude={latitude}"
        f"&longitude={longitude}"
        "&daily=temperature_2m_max"
        "&timezone=auto"
    )

    response = requests.get(url, timeout=10)
    response.raise_for_status()

    data = response.json()

    return {
        "timezone": data.get("timezone"),
        "dates": data["daily"]["time"],
        "max_temperatures": data["daily"]["temperature_2m_max"]
    }

In [9]:
weather = get_weather(latitude=15.2993, longitude=74.1240)
weather

{'timezone': 'Asia/Kolkata',
 'dates': ['2025-12-23',
  '2025-12-24',
  '2025-12-25',
  '2025-12-26',
  '2025-12-27',
  '2025-12-28',
  '2025-12-29'],
 'max_temperatures': [32.7, 32.8, 33.0, 33.0, 33.5, 33.6, 32.5]}

## **Flight Search Tool (Flights JSON)**

In [12]:
import os

os.makedirs("data", exist_ok=True)
os.listdir(".")

['.config', 'data', 'tools', 'sample_data']

In [13]:
!ls

data  sample_data  tools


In [14]:
import os

os.listdir("data")

['places.json', 'flights.json', 'hotels.json']

In [11]:
import os

os.makedirs("data", exist_ok=True)

print("data folder created")

data folder created


In [15]:
import json

with open("data/flights.json", "r", encoding="utf-8") as f:
    flights_data = json.load(f)

len(flights_data)

30

In [16]:
def search_flights(source: str, destination: str):
    """
    Return the cheapest flight for a given route.
    """
    matches = [
        flight for flight in flights_data
        if flight["from"].lower() == source.lower()
        and flight["to"].lower() == destination.lower()
    ]

    if not matches:
        return "No flights found for this route."

    cheapest_flight = min(matches, key=lambda x: x["price"])
    return cheapest_flight

In [17]:
search_flights("Hyderabad", "Delhi")

{'flight_id': 'FL0001',
 'airline': 'IndiGo',
 'from': 'Hyderabad',
 'to': 'Delhi',
 'departure_time': '2025-01-04T11:32:00',
 'arrival_time': '2025-01-04T15:32:00',
 'price': 2907}

## **Hotel & Places Recommendation Tools**

In [18]:
import json

with open("data/hotels.json", "r", encoding="utf-8") as f:
    hotels_data = json.load(f)

len(hotels_data)

40

In [19]:
def recommend_hotel(city: str):
    """
    Return the best hotel based on highest stars and lowest price.
    """
    matches = [
        hotel for hotel in hotels_data
        if hotel["city"].lower() == city.lower()
    ]

    if not matches:
        return "No hotels found for this city."

    best_hotel = sorted(
        matches,
        key=lambda x: (-x["stars"], x["price_per_night"])
    )[0]

    return best_hotel

In [20]:
recommend_hotel("Delhi")

{'hotel_id': 'HOT0002',
 'name': 'Comfort Suites',
 'city': 'Delhi',
 'stars': 5,
 'price_per_night': 3650,
 'amenities': ['gym', 'breakfast', 'wifi', 'parking']}

In [21]:
with open("data/places.json", "r", encoding="utf-8") as f:
    places_data = json.load(f)

len(places_data)

40

In [22]:
def recommend_places(city: str, top_n: int = 3):
    """
    Return top-rated places to visit in a city.
    """
    matches = [
        place for place in places_data
        if place["city"].lower() == city.lower()
    ]

    if not matches:
        return "No places found for this city."

    top_places = sorted(
        matches,
        key=lambda x: x["rating"],
        reverse=True
    )[:top_n]

    return top_places

In [23]:
recommend_places("Delhi")

[{'place_id': 'PLC0001',
  'name': 'Famous Fort',
  'city': 'Delhi',
  'type': 'lake',
  'rating': 4.6},
 {'place_id': 'PLC0004',
  'name': 'Popular Museum',
  'city': 'Delhi',
  'type': 'lake',
  'rating': 4.5},
 {'place_id': 'PLC0002',
  'name': 'Beautiful Temple',
  'city': 'Delhi',
  'type': 'temple',
  'rating': 4.2}]

## **Budget Estimation Tool**

In [24]:
def estimate_budget(
    flight_price: int,
    hotel_price_per_night: int,
    nights: int,
    daily_local_cost: int = 1000
):
    """
    Estimate total trip budget.
    """
    hotel_cost = hotel_price_per_night * nights
    local_cost = daily_local_cost * nights

    total_cost = flight_price + hotel_cost + local_cost

    return {
        "flight_cost": flight_price,
        "hotel_cost": hotel_cost,
        "local_cost": local_cost,
        "total_cost": total_cost
    }

In [25]:
estimate_budget(
    flight_price=2907,
    hotel_price_per_night=3650,
    nights=2
)

{'flight_cost': 2907,
 'hotel_cost': 7300,
 'local_cost': 2000,
 'total_cost': 12207}

## **Core Agent Logic (Tool Orchestration)**

In [38]:
def travel_agent(
    source_city: str,
    destination_city: str,
    latitude: float,
    longitude: float,
    nights: int
):
    """
    Core agent that plans a trip using available tools.
    """

    # Flight
    flight = search_flights(source_city, destination_city)
    if isinstance(flight, str):
        return flight

    # Hotel
    hotel = recommend_hotel(destination_city)
    if isinstance(hotel, str):
        return hotel

    # Places
    places = recommend_places(destination_city)

    # Weather
    weather = get_weather(latitude, longitude)

    # Budget
    budget = estimate_budget(
        flight_price=flight["price"],
        hotel_price_per_night=hotel["price_per_night"],
        nights=nights
    )

    explanations = explain_selections(flight, hotel, places)
    day_wise_plan = create_day_wise_itinerary(places, nights)

    return {
        "route": f"{source_city} → {destination_city}",
        "flight": flight,
        "hotel": hotel,
        "places": places,
        "day_wise_itinerary": day_wise_plan,
        "weather": weather,
        "budget": budget,
        "explanations": explanations
    }

In [39]:
travel_agent(
    "Hyderabad",
    "Delhi",
    28.6139,
    77.2090,
    2
)

{'route': 'Hyderabad → Delhi',
 'flight': {'flight_id': 'FL0001',
  'airline': 'IndiGo',
  'from': 'Hyderabad',
  'to': 'Delhi',
  'departure_time': '2025-01-04T11:32:00',
  'arrival_time': '2025-01-04T15:32:00',
  'price': 2907},
 'hotel': {'hotel_id': 'HOT0002',
  'name': 'Comfort Suites',
  'city': 'Delhi',
  'stars': 5,
  'price_per_night': 3650,
  'amenities': ['gym', 'breakfast', 'wifi', 'parking']},
 'places': [{'place_id': 'PLC0001',
   'name': 'Famous Fort',
   'city': 'Delhi',
   'type': 'lake',
   'rating': 4.6},
  {'place_id': 'PLC0004',
   'name': 'Popular Museum',
   'city': 'Delhi',
   'type': 'lake',
   'rating': 4.5},
  {'place_id': 'PLC0002',
   'name': 'Beautiful Temple',
   'city': 'Delhi',
   'type': 'temple',
   'rating': 4.2}],
 'day_wise_itinerary': {'Day 1': 'Famous Fort', 'Day 2': 'Popular Museum'},
 'weather': {'timezone': 'Asia/Kolkata',
  'dates': ['2025-12-23',
   '2025-12-24',
   '2025-12-25',
   '2025-12-26',
   '2025-12-27',
   '2025-12-28',
   '2025-12

In [27]:
travel_agent(
    source_city="Hyderabad",
    destination_city="Delhi",
    latitude=28.6139,
    longitude=77.2090,
    nights=2
)

{'route': 'Hyderabad → Delhi',
 'flight_selected': {'flight_id': 'FL0001',
  'airline': 'IndiGo',
  'from': 'Hyderabad',
  'to': 'Delhi',
  'departure_time': '2025-01-04T11:32:00',
  'arrival_time': '2025-01-04T15:32:00',
  'price': 2907},
 'hotel_selected': {'hotel_id': 'HOT0002',
  'name': 'Comfort Suites',
  'city': 'Delhi',
  'stars': 5,
  'price_per_night': 3650,
  'amenities': ['gym', 'breakfast', 'wifi', 'parking']},
 'places_to_visit': [{'place_id': 'PLC0001',
   'name': 'Famous Fort',
   'city': 'Delhi',
   'type': 'lake',
   'rating': 4.6},
  {'place_id': 'PLC0004',
   'name': 'Popular Museum',
   'city': 'Delhi',
   'type': 'lake',
   'rating': 4.5},
  {'place_id': 'PLC0002',
   'name': 'Beautiful Temple',
   'city': 'Delhi',
   'type': 'temple',
   'rating': 4.2}],
 'weather_forecast': {'timezone': 'Asia/Kolkata',
  'dates': ['2025-12-23',
   '2025-12-24',
   '2025-12-25',
   '2025-12-26',
   '2025-12-27',
   '2025-12-28',
   '2025-12-29'],
  'max_temperatures': [22.2, 20.0

In [32]:
def explain_selections(flight, hotel, places):
    return {
        "flight_reason": f"Selected because it is the cheapest available option at ₹{flight['price']}.",
        "hotel_reason": f"Selected based on highest rating ({hotel['stars']}⭐) with a reasonable nightly price.",
        "places_reason": "Selected based on highest user ratings and popularity."
    }

In [33]:
def create_day_wise_itinerary(places, days):
    itinerary = {}
    for i in range(days):
        if i < len(places):
            itinerary[f"Day {i+1}"] = places[i]["name"]
        else:
            itinerary[f"Day {i+1}"] = "Leisure / Local exploration"
    return itinerary

## **Final Completion (Explanations, Itinerary, Streamlit)**

In [34]:
def explain_selections(flight, hotel, places):
    return {
        "flight_reason": f"Selected because it is the cheapest available flight at ₹{flight['price']}.",
        "hotel_reason": f"Selected due to highest rating ({hotel['stars']}⭐) with competitive pricing.",
        "places_reason": "Selected based on highest user ratings and popularity."
    }

In [35]:
def create_day_wise_itinerary(places, nights):
    itinerary = {}
    for i in range(nights):
        if i < len(places):
            itinerary[f"Day {i+1}"] = places[i]["name"]
        else:
            itinerary[f"Day {i+1}"] = "Leisure / Local exploration"
    return itinerary

In [40]:
!pip install streamlit pyngrok

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 68.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 87.4 MB/s eta 0:00:00


In [41]:
%%writefile app.py
import streamlit as st

st.set_page_config(
    page_title="Agentic AI-Based Travel Planning Assistant",
    layout="wide"
)

st.title("Agentic AI-Based Travel Planning Assistant")

source = st.text_input("Source City", "Hyderabad")
destination = st.text_input("Destination City", "Delhi")
nights = st.number_input("Number of Nights", min_value=1, max_value=10, value=2)

latitude = st.number_input("Destination Latitude", value=28.6139)
longitude = st.number_input("Destination Longitude", value=77.2090)

if st.button("Plan My Trip"):
    result = travel_agent(
        source,
        destination,
        latitude,
        longitude,
        nights
    )
    st.subheader("Trip Output")
    st.json(result)

Writing app.py


In [ ]:
from pyngrok import ngrok

# Start Streamlit server in background
!streamlit run app.py &

# Create public URL
public_url = ngrok.connect(8501)
public_url




  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.50.190.230:8501

